# ⚡ Analyse de la Consommation Électrique avec un LSTM

## Présentation du projet

Dans ce notebook, nous allons analyser un dataset réel de consommation électrique d'un foyer, et construire un modèle de **deep learning** (LSTM) pour prédire la consommation.

### 🎯 Ce que vous allez apprendre :
- Importer et manipuler des données de séries temporelles avec **pandas**
- Gérer les **valeurs manquantes**
- Visualiser des données avec **matplotlib** et **seaborn**
- Construire et entraîner un modèle **LSTM** avec Keras/TensorFlow

### 📊 Le Dataset
Le dataset contient la consommation électrique d'un foyer entre 2006 et 2010, avec une mesure toutes les minutes.

---
> ⚠️ **Rappel** : Assurez-vous d'avoir activé le GPU dans Google Colab :
> `Runtime > Change runtime type > Hardware accelerator > GPU`

---
## 📦 PARTIE 1 — Import des données et exploration initiale

**Objectif :** Charger les données et comprendre leur structure.

On commence toujours par importer les bibliothèques nécessaires, puis on charge les données pour les explorer.

In [ ]:
# ── Étape 1 : Importer les bibliothèques ──────────────────────────────────────
# pandas  → manipulation de tableaux de données
# numpy   → calculs numériques
# matplotlib / seaborn → visualisations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Pour un affichage soigné des graphiques dans le notebook
%matplotlib inline
sns.set_theme(style='darkgrid')  # Thème esthétique pour seaborn

print("✅ Bibliothèques importées avec succès !")

In [ ]:
# ── Étape 2 : Télécharger et décompresser le dataset ─────────────────────────
# Le dataset est disponible sur GitHub sous forme de fichier .zip

import urllib.request
import zipfile
import os

url = "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%206/W6D4/household_power_consumption.zip"
zip_path = "household_power_consumption.zip"

# Téléchargement
print("📥 Téléchargement du dataset...")
urllib.request.urlretrieve(url, zip_path)

# Décompression
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall('.')

print("✅ Dataset téléchargé et décompressé !")
print("Fichiers disponibles :", os.listdir('.'))

In [ ]:
# ── Étape 3 : Charger le dataset ─────────────────────────────────────────────
# Le fichier utilise ';' comme séparateur (pas la virgule habituelle)
# On indique aussi que '?' représente les valeurs manquantes

df = pd.read_csv(
    'household_power_consumption.txt',
    sep=';',                    # séparateur de colonnes
    na_values='?',              # '?' = valeur manquante
    low_memory=False            # évite les avertissements sur les types
)

# Combiner 'Date' et 'Time' en un seul index datetime
# Cela est essentiel pour travailler avec des séries temporelles !
df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True)
df.set_index('datetime', inplace=True)
df.drop(columns=['Date', 'Time'], inplace=True)

print("✅ Dataset chargé !")

In [ ]:
# ── Étape 4 : Explorer les premières lignes ───────────────────────────────────
# .head() affiche les 5 premières lignes → permet de voir la structure

print("📋 Aperçu des premières lignes :")
df.head()

In [ ]:
# ── Étape 5 : Vérifier les types et la forme du dataset ──────────────────────

print(f"📐 Forme du dataset : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print()
print("🔍 Types de données par colonne :")
print(df.dtypes)
print()
print("📊 Statistiques descriptives :")
df.describe()

---
## 🧹 PARTIE 2 — Gestion des valeurs manquantes

**Objectif :** Détecter et traiter les valeurs manquantes.

Les données réelles contiennent souvent des trous (pannes, erreurs de capteur…). Il faut les identifier et les remplacer par des valeurs raisonnables.  
Ici, on utilise la **moyenne de la colonne** comme valeur de remplacement.

In [ ]:
# ── Étape 1 : Identifier les valeurs manquantes ───────────────────────────────
# .isnull().sum() compte le nombre de NaN par colonne

missing = df.isnull().sum()
print("❓ Nombre de valeurs manquantes par colonne :")
print(missing)
print()

# Calcul du pourcentage de données manquantes
pct_missing = (missing / len(df)) * 100
print("📉 Pourcentage manquant :")
print(pct_missing.round(2))

In [ ]:
# ── Étape 2 : Remplir les valeurs manquantes par la moyenne ──────────────────
# Pourquoi la moyenne ? C'est une stratégie simple et neutre :
# elle ne biaise pas la distribution des données.

df.fillna(df.mean(numeric_only=True), inplace=True)

print("✅ Valeurs manquantes remplies par la moyenne de chaque colonne.")

In [ ]:
# ── Étape 3 : Vérifier qu'il ne reste plus de valeurs manquantes ─────────────

remaining = df.isnull().sum().sum()
print(f"🔎 Valeurs manquantes restantes : {remaining}")

if remaining == 0:
    print("✅ Le dataset est propre — aucune valeur manquante !")
else:
    print("⚠️ Il reste des valeurs manquantes, vérifiez le code.")

---
## 📊 PARTIE 3 — Visualisation des données

**Objectif :** Comprendre la dynamique des données grâce à des graphiques.

On va **rééchantillonner** les données à la journée (au lieu de la minute) pour avoir une vision plus claire des tendances.  
→ `resample('D')` regroupe les mesures par jour.

In [ ]:
# ── Graphique 1 : Somme et Moyenne journalières de la puissance active ────────
# La 'Global_active_power' est la mesure principale de consommation électrique

daily_power = df['Global_active_power'].resample('D')  # Regroupement par jour

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# --- Somme journalière ---
daily_power.sum().plot(ax=axes[0], color='steelblue', linewidth=0.8)
axes[0].set_title('Consommation totale journalière (Global Active Power — Somme)', fontsize=13)
axes[0].set_ylabel('kW·h')

# --- Moyenne journalière ---
daily_power.mean().plot(ax=axes[1], color='darkorange', linewidth=0.8)
axes[1].set_title('Consommation moyenne journalière (Global Active Power — Moyenne)', fontsize=13)
axes[1].set_ylabel('kW')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

print("💡 On observe une saisonnalité : la consommation est plus haute en hiver (chauffage).")

In [ ]:
# ── Graphique 2 : Moyenne et Écart-type de l'intensité ────────────────────────
# L'intensité (Global_intensity) mesure le courant en ampères
# L'écart-type montre la variabilité (plus il est grand, plus la conso fluctue)

daily_intensity = df['Global_intensity'].resample('D')
mean_intensity  = daily_intensity.mean()
std_intensity   = daily_intensity.std()

fig, ax = plt.subplots(figsize=(14, 5))

# Ligne de la moyenne
ax.plot(mean_intensity.index, mean_intensity, color='teal', linewidth=0.9, label='Moyenne')

# Zone ombrée = moyenne ± 1 écart-type
ax.fill_between(
    mean_intensity.index,
    mean_intensity - std_intensity,
    mean_intensity + std_intensity,
    alpha=0.25, color='teal', label='± 1 écart-type'
)

ax.set_title("Intensité globale journalière — Moyenne et écart-type", fontsize=13)
ax.set_ylabel("Ampères (A)")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

print("💡 Les zones sombres indiquent des périodes de forte variabilité dans la consommation.")

---
## ⚙️ PARTIE 4 — Prétraitement des données pour le LSTM

**Objectif :** Préparer les données dans le format attendu par un réseau LSTM.

### Pourquoi prétraiter ?
1. **Normalisation** : Les LSTM sont sensibles aux échelles. On ramène toutes les valeurs entre 0 et 1.
2. **Découpage train/test** : On garde 80 % pour l'entraînement, 20 % pour l'évaluation.
3. **Reshape 3D** : Les LSTM attendent des données en forme `(samples, timesteps, features)`.

⚠️ On travaille sur les **données journalières** pour réduire le volume de calcul.

In [ ]:
# ── Étape 1 : Réduire à la granularité journalière ────────────────────────────
# On garde uniquement Global_active_power pour la prédiction

daily_df = df.resample('D').mean(numeric_only=True)

print(f"📅 Dataset journalier : {daily_df.shape[0]} jours × {daily_df.shape[1]} colonnes")
daily_df.head()

In [ ]:
# ── Étape 2 : Normalisation (MinMaxScaler) ────────────────────────────────────
# MinMaxScaler ramène chaque valeur entre 0 et 1 :
#   x_norm = (x - x_min) / (x_max - x_min)
# C'est crucial pour aider le réseau à converger rapidement.

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 1))

# On sélectionne toutes les colonnes numériques
data_scaled = scaler.fit_transform(daily_df)

print(f"✅ Données normalisées — min: {data_scaled.min():.2f}, max: {data_scaled.max():.2f}")
print(f"   Forme : {data_scaled.shape}")

In [ ]:
# ── Étape 3 : Découpage train / test ─────────────────────────────────────────
# Convention : 80 % pour l'entraînement, 20 % pour le test
# IMPORTANT : on ne mélange PAS les données (c'est une série temporelle !
#             les données du futur ne doivent pas contaminer l'entraînement)

split = int(len(data_scaled) * 0.8)

train_data = data_scaled[:split]
test_data  = data_scaled[split:]

print(f"🔀 Découpage : {split} jours d'entraînement | {len(data_scaled) - split} jours de test")

In [ ]:
# ── Étape 4 : Créer des séquences pour le LSTM ───────────────────────────────
# Un LSTM prédit la valeur au temps t en regardant les 'look_back' valeurs précédentes.
# Exemple avec look_back=7 : on utilise 7 jours passés pour prédire le 8ème.

def create_sequences(data, look_back=30):
    """
    Transforme un tableau 2D en séquences pour LSTM.
    
    Args:
        data      : tableau numpy normalisé
        look_back : nombre de pas de temps passés à regarder
    
    Returns:
        X : shape (n_samples, look_back, n_features)
        y : shape (n_samples,)  — valeur de Global_active_power à prédire
    """
    X, y = [], []
    for i in range(look_back, len(data)):
        X.append(data[i - look_back:i, :])   # fenêtre de look_back jours
        y.append(data[i, 0])                  # colonne 0 = Global_active_power
    return np.array(X), np.array(y)

LOOK_BACK = 30  # On regarde les 30 derniers jours

X_train, y_train = create_sequences(train_data, LOOK_BACK)
X_test,  y_test  = create_sequences(test_data,  LOOK_BACK)

print("📐 Formes des données :")
print(f"   X_train : {X_train.shape}  → (échantillons, timesteps, features)")
print(f"   y_train : {y_train.shape}")
print(f"   X_test  : {X_test.shape}")
print(f"   y_test  : {y_test.shape}")

---
## 🧠 PARTIE 5 — Construction du modèle LSTM

**Objectif :** Définir l'architecture du réseau de neurones.

### Qu'est-ce qu'un LSTM ?
Le **Long Short-Term Memory** est un type de réseau de neurones récurrent (RNN) conçu pour apprendre des dépendances à long terme dans les séquences.  
Il est idéal pour les **séries temporelles** car il se souvient des informations passées pertinentes.

### Architecture choisie :
```
Input (30 jours × 7 features)
    ↓
LSTM (64 unités, retourne la séquence)
    ↓
Dropout (20%) ← évite le surapprentissage
    ↓
LSTM (32 unités)
    ↓
Dropout (20%)
    ↓
Dense (1) ← prédiction finale
```

In [ ]:
# ── Import des bibliothèques deep learning ────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print(f"🔧 TensorFlow version : {tf.__version__}")
print(f"🖥️  GPU disponible : {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ── Définition de l'architecture LSTM ────────────────────────────────────────

n_features = X_train.shape[2]  # Nombre de colonnes (features)

model = Sequential([
    # Couche 1 : LSTM avec 64 neurones
    # return_sequences=True → passe toute la séquence à la couche suivante
    LSTM(64, return_sequences=True, input_shape=(LOOK_BACK, n_features)),
    Dropout(0.2),  # Désactive 20% des neurones aléatoirement → régularisation

    # Couche 2 : LSTM avec 32 neurones
    # return_sequences=False → ne retourne que le dernier état
    LSTM(32, return_sequences=False),
    Dropout(0.2),

    # Couche de sortie : 1 neurone = 1 valeur prédite
    Dense(1)
])

# ── Compilation ───────────────────────────────────────────────────────────────
# loss='mse'  → Mean Squared Error, classique pour la régression
# optimizer='adam' → algorithme d'optimisation adaptatif, très utilisé
model.compile(loss='mse', optimizer='adam', metrics=['mae'])

# Résumé de l'architecture
model.summary()

---
## 🚀 PARTIE 6 — Entraînement et Évaluation du modèle

**Objectif :** Entraîner le modèle et analyser ses performances.

### EarlyStopping
On utilise un **arrêt précoce** : si la validation loss ne s'améliore plus pendant 10 epochs consécutives, l'entraînement s'arrête automatiquement. Cela évite le surapprentissage et économise du temps.

In [ ]:
# ── Entraînement du modèle ────────────────────────────────────────────────────

# EarlyStopping : arrête si pas d'amélioration après 10 epochs
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True  # Conserve les meilleurs poids
)

history = model.fit(
    X_train, y_train,
    epochs=100,          # Maximum 100 epochs (EarlyStopping peut arrêter avant)
    batch_size=32,       # Nombre d'échantillons traités en parallèle
    validation_split=0.1,# 10% du train sert à valider à chaque epoch
    callbacks=[early_stop],
    verbose=1            # Affiche la progression
)

print(f"\n✅ Entraînement terminé après {len(history.history['loss'])} epochs.")

In [ ]:
# ── Évaluation sur le jeu de test ────────────────────────────────────────────
# Le jeu de test est constitué de données que le modèle n'a JAMAIS vues.
# C'est la mesure de performance la plus honnête.

test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)

print("📊 Performance sur le jeu de TEST :")
print(f"   MSE (Mean Squared Error)       : {test_loss:.4f}")
print(f"   MAE (Mean Absolute Error)      : {test_mae:.4f}")
print()
print("💡 Rappel : les valeurs sont normalisées (entre 0 et 1).")
print("   Un MAE de 0.05 signifie une erreur d'environ 5% de l'échelle totale.")

In [ ]:
# ── Graphique de la Loss (entraînement vs validation) ─────────────────────────
# Ce graphique est essentiel pour diagnostiquer :
#   - Surapprentissage (overfitting) : val_loss >> train_loss
#   - Sous-apprentissage (underfitting) : les deux losses restent hautes
#   - Bon apprentissage : les deux losses descendent ensemble

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(history.history['loss'],     label='Train Loss',      color='royalblue',  linewidth=2)
ax.plot(history.history['val_loss'], label='Validation Loss', color='tomato', linewidth=2, linestyle='--')

ax.set_title("Évolution de la Loss pendant l'entraînement", fontsize=14)
ax.set_xlabel("Epochs")
ax.set_ylabel("MSE Loss")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualisation des prédictions vs valeurs réelles ─────────────────────────
# On dé-normalise les valeurs pour les remettre à l'échelle originale (kW)

# Prédictions sur le jeu de test
y_pred_scaled = model.predict(X_test)

# Pour inverser la normalisation, on doit reconstruire un tableau complet
# (MinMaxScaler a été ajusté sur toutes les colonnes)
def inverse_transform_col0(scaler, values, n_features):
    """Inverse la normalisation pour la colonne 0 uniquement."""
    dummy = np.zeros((len(values), n_features))
    dummy[:, 0] = values.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

y_pred_real = inverse_transform_col0(scaler, y_pred_scaled, n_features)
y_test_real = inverse_transform_col0(scaler, y_test.reshape(-1,1), n_features)

# Graphique
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_test_real,  label='Valeurs réelles',  color='steelblue', linewidth=1.5)
ax.plot(y_pred_real,  label='Prédictions LSTM', color='tomato',    linewidth=1.5, linestyle='--')

ax.set_title("Prédictions LSTM vs Valeurs réelles (jeu de test)", fontsize=14)
ax.set_xlabel("Jours")
ax.set_ylabel("Global Active Power (kW)")
ax.legend()
plt.tight_layout()
plt.show()

# Erreur RMSE en unités réelles
rmse = np.sqrt(np.mean((y_test_real - y_pred_real)**2))
print(f"📏 RMSE en unités réelles : {rmse:.4f} kW")

---
## 🎓 Conclusion

Bravo ! Vous avez réalisé un pipeline complet de Machine Learning sur des données temporelles réelles :

| Étape | Ce qu'on a fait |
|-------|----------------|
| **Import** | Chargé un dataset de 2M+ lignes avec pandas |
| **Nettoyage** | Remplacé les `?` par la moyenne de chaque colonne |
| **Visualisation** | Identifié une saisonnalité dans la consommation |
| **Prétraitement** | Normalisé + créé des séquences (fenêtre glissante) |
| **Modèle** | Construit un LSTM à 2 couches avec Dropout |
| **Évaluation** | Mesuré la performance et visualisé les prédictions |

### 🚀 Pour aller plus loin :
- Augmenter `LOOK_BACK` pour capturer des saisonnalités plus longues
- Ajouter des features temporelles (mois, jour de la semaine)
- Essayer un modèle plus profond ou un Transformer